# GNN
*   For the GNN model, only the code for computation on the test set is currently provided (excluding the code for computation on the training set).

In [1]:
# PC
import GNN
import numpy as np
import pandas as pd
import torch
from rdkit import Chem
torch.set_num_threads(16)
seed_list=[42, 403138084, 573583737, 604087939, 643026125, 661892772, 866264613, 1509598813, 1605674636, 1687343257]
df = pd.read_csv('../testdata/Pc_testset.csv')
need_drop = []
for index, row in df.iterrows():
    try:
        GNN.mol_to_pyg(Chem.MolFromSmiles(row['smiles']))
    except Exception as e:
        need_drop.append(index)
        print(f"Error processing SMILES: {row['smiles']} at index {index}, error: {str(e)}")
df = df.drop(need_drop).reset_index(drop=True)
X_train = df['smiles'].to_list()
P_mean = 14.989541546234026
P_std = 0.43770555747724915
input_loader = GNN.preprocess(X_train)

models = []
hidden_dim = 48 
num_gnn_layers = 2
num_hidden_layers = 6
gnn_heads = 7
pooling_heads = 4
dropout = 0.16225697809516074
for seed in seed_list:
    model = GNN.GNN(node_dim = 38, edge_dim = 9, 
                conv_dim = 32, hidden_dim = hidden_dim, num_hidden_layers=num_hidden_layers, dropout=dropout,
                num_gnn_layers = num_gnn_layers, gnn_heads = gnn_heads, pooling_heads = pooling_heads)
    model.load_state_dict(torch.load(f'../model/GNN_PC_{seed}.pth', weights_only=True, map_location=torch.device('cpu')))
    model.eval()
    # model = torch.jit.script(model)
    models.append(model)

results = []
with torch.no_grad():
    for model in models:
        y_pred = []
        for X in input_loader:
            y_pred.extend(model(X.x, X.edge_index, X.edge_attr, X.numHDonors, X.numHAcceptors, X.batch).numpy())
        results.append(np.array(y_pred))
results = np.array(results)
results = results * P_std + P_mean
y_pred_mean = np.mean(results, axis=0).flatten()
y_std = np.std(results, axis=0)
df = df[list(df.columns)[:11]]
df['PC_pred'] = np.exp(y_pred_mean)
df['PC_std'] = y_std
df = df[['Name', 'PC_pred', 'PC_std']]
df.to_csv('./test_Pred/GNN-PC.csv', index=False)

Error processing SMILES: [2H]S[2H] at index 17, error: input H not in allowable set['B', 'Se', 'Sb', 'In', 'Cl', 'I', 'Ga', 'P', 'N', 'Hg', 'Ge', 'Al', 'C', 'Sn', 'V', 'Br', 'O', 'S', 'As', 'Pb', 'Si', 'F', 'Ti']:
Error processing SMILES: [2H]Cl at index 74, error: input H not in allowable set['B', 'Se', 'Sb', 'In', 'Cl', 'I', 'Ga', 'P', 'N', 'Hg', 'Ge', 'Al', 'C', 'Sn', 'V', 'Br', 'O', 'S', 'As', 'Pb', 'Si', 'F', 'Ti']:
Error processing SMILES: [2H]C([2H])([2H])[2H] at index 90, error: input H not in allowable set['B', 'Se', 'Sb', 'In', 'Cl', 'I', 'Ga', 'P', 'N', 'Hg', 'Ge', 'Al', 'C', 'Sn', 'V', 'Br', 'O', 'S', 'As', 'Pb', 'Si', 'F', 'Ti']:
Error processing SMILES: FS(F)(F)(F)(F)F at index 93, error: input SP3D2 not in allowable set[rdkit.Chem.rdchem.HybridizationType.S, rdkit.Chem.rdchem.HybridizationType.SP, rdkit.Chem.rdchem.HybridizationType.SP2, rdkit.Chem.rdchem.HybridizationType.SP3]:


# ANN

## training and validation

In [2]:
import numpy as np
import pandas as pd
import torch
from model import Crit_model
torch.set_num_threads(16)
seed_list=[42, 403138084, 573583737, 604087939, 643026125, 661892772, 866264613, 1509598813, 1605674636, 1687343257]
df_meanstd = pd.read_csv('../database/Crit-Pc-MeanStd.csv')
mean = df_meanstd['mean'].values[:76-11]
std = df_meanstd['std'].values[:76-11]
df = pd.read_csv('../database/Crit-Pc.csv')
X_train = df.iloc[:, 2:].values
X_train = (X_train - mean) / std
P_mean = 14.989541546234026
P_std = 0.43770555747724915

models = []
hidden_dim = 44
hidden_num = 2
hidden_type = "1_1"
activation = "gelu"
dropout = 0.0
for seed in seed_list:
    model = Crit_model(input_size=X_train.shape[1], 
                       output_size=1,
                       hidden_size=hidden_dim, 
                       hidden_num=hidden_num, 
                       hidden_type=hidden_type, 
                       activation=activation, 
                       dropout=dropout)
    model.load_state_dict(torch.load(f'../model/ANN_PC2D_{seed}.pth', weights_only=True, map_location=torch.device('cpu')))
    model.eval()
    model = torch.jit.script(model)
    models.append(model)

X_train = torch.from_numpy(X_train).float()
results = []
with torch.no_grad():
    for model in models:
        y_pred = model(X_train).numpy()
        results.append(y_pred)
results = np.array(results)
results = results * P_std + P_mean
y_pred_mean = np.mean(results, axis=0).flatten()
y_std = np.std(results, axis=0)
df['PC2D_pred'] = np.exp(y_pred_mean)
df['PC2D_std'] = y_std
df = df[['material or substance name', 'PC2D_pred', 'PC2D_std']]
df.to_csv('./training_Pred/ANN-PC.csv', index=False)

## test

In [3]:
# PC2D test
import numpy as np
import pandas as pd
import torch
from model import Crit_model
torch.set_num_threads(16)
seed_list=[42, 403138084, 573583737, 604087939, 643026125, 661892772, 866264613, 1509598813, 1605674636, 1687343257]
df_meanstd = pd.read_csv('../database/Crit-Pc-MeanStd.csv')
mean = df_meanstd['mean'].values
std = df_meanstd['std'].values
df = pd.read_csv('../testdata/Pc_testset.csv')
X_train = df.iloc[:, 2:].values
X_train = (X_train - mean) / std
P_mean = 14.989541546234026
P_std = 0.43770555747724915

models_2D = []
hidden_dim = 44
hidden_num = 2
hidden_type = "1_1"
activation = "gelu"
dropout = 0.0
for seed in seed_list:
    model = Crit_model(input_size=76-11, 
                       output_size=1,
                       hidden_size=hidden_dim, 
                       hidden_num=hidden_num, 
                       hidden_type=hidden_type, 
                       activation=activation, 
                       dropout=dropout)
    model.load_state_dict(torch.load(f'../model/ANN_PC2D_{seed}.pth', weights_only=True, map_location=torch.device('cpu')))
    model.eval()
    model = torch.jit.script(model)
    models_2D.append(model)

models_3D = []
hidden_dim = 52 
hidden_num = 4 
hidden_type = "1_1"
activation = "gelu"
dropout = 0.0
for seed in seed_list:
    model = Crit_model(input_size=X_train.shape[1], 
                       output_size=1,
                       hidden_size=hidden_dim, 
                       hidden_num=hidden_num, 
                       hidden_type=hidden_type, 
                       activation=activation, 
                       dropout=dropout)
    model.load_state_dict(torch.load(f'../model/ANN_PC_{seed}.pth', weights_only=True, map_location=torch.device('cpu')))
    model.eval()
    model = torch.jit.script(model)
    models_3D.append(model)

X_train = torch.from_numpy(X_train).float()

results_3D = []
with torch.no_grad():
    for model in models_3D:
        y_pred = model(X_train).numpy()
        results_3D.append(y_pred)
results_3D = np.array(results_3D)
results_3D = results_3D * P_std + P_mean
y_pred_mean_3D = np.mean(results_3D, axis=0).flatten()
y_std_3D = np.std(results_3D, axis=0)

results_2D = []
X_train = X_train[:, :76-11]
with torch.no_grad():
    for model in models_2D:
        y_pred = model(X_train).numpy()
        results_2D.append(y_pred)
results_2D = np.array(results_2D)
results_2D = results_2D * P_std + P_mean
y_pred_mean_2D = np.mean(results_2D, axis=0).flatten()
y_std_2D = np.std(results_2D, axis=0)

df['PC3D_pred'] = np.exp(y_pred_mean_3D)
df['PC3D_std'] = y_std_3D
df['PC2D_pred'] = np.exp(y_pred_mean_2D)
df['PC2D_std'] = y_std_2D

df = df[['Name', 'PC3D_pred', 'PC3D_std', 'PC2D_pred', 'PC2D_std']]
df.to_csv('./test_Pred/ANN-PC.csv', index=False)

# GPR

## training and validation

In [4]:
import numpy as np
import pandas as pd
import torch
import joblib
torch.set_num_threads(16)
df_meanstd = pd.read_csv('../database/Crit-Pc-MeanStd.csv')
mean = df_meanstd['mean'].values[:76-11]
std = df_meanstd['std'].values[:76-11]
P_mean = 14.989541546234026
P_std = 0.43770555747724915
df = pd.read_csv('../database/Crit-Pc.csv')
X_train = df.iloc[:, 2:].values
X_train = (X_train - mean) / std
model2d = joblib.load('../model/gpr_pc_2d.pkl')
y_pred_2d = model2d.predict(X_train) * P_std + P_mean
df['PC2D_pred'] = np.exp(y_pred_2d)
df = df[['material or substance name', 'PC2D_pred']]
df.to_csv('./training_Pred/GPR-PC.csv', index=False)

## test

In [5]:
import numpy as np
import pandas as pd
import torch
import joblib
torch.set_num_threads(16)
df_meanstd = pd.read_csv('../database/Crit-Pc-MeanStd.csv')
mean = df_meanstd['mean'].values
std = df_meanstd['std'].values
P_mean = 14.989541546234026
P_std = 0.43770555747724915
df = pd.read_csv('../testdata/Pc_testset.csv')
X_train = df.iloc[:, 2:].values
X_train = (X_train - mean) / std
model2d = joblib.load('../model/gpr_pc_2d.pkl')
model3d = joblib.load('../model/gpr_pc_3d.pkl')
y_pred_3d = model3d.predict(X_train) * P_std + P_mean
y_pred_2d = model2d.predict(X_train[:, :76-11]) * P_std + P_mean
df['PC3D_pred'] = np.exp(y_pred_3d)
df['PC2D_pred'] = np.exp(y_pred_2d)
df = df[['Name', 'PC2D_pred', 'PC3D_pred']]
df.to_csv('./test_Pred/GPR-PC.csv', index=False)

# RF

## training and validation

In [6]:
import numpy as np
import pandas as pd
import torch
import joblib
torch.set_num_threads(16)
df_meanstd = pd.read_csv('../database/Crit-Pc-MeanStd.csv')
mean = df_meanstd['mean'].values[:76-11]
std = df_meanstd['std'].values[:76-11]
df = pd.read_csv('../database/Crit-Pc.csv')
X_train = df.iloc[:, 2:].values
X_train = (X_train - mean) / std
model2d = joblib.load('../model/rf_pc_2d.pkl')
y_pred_2d = model2d.predict(X_train)
df['PC2D_pred'] = np.exp(y_pred_2d)
df = df[['material or substance name', 'PC2D_pred']]
df.to_csv('./training_Pred/RF-PC.csv', index=False)

## test

In [7]:
import numpy as np
import pandas as pd
import torch
import joblib
torch.set_num_threads(16)
df_meanstd = pd.read_csv('../database/Crit-Pc-MeanStd.csv')
mean = df_meanstd['mean'].values
std = df_meanstd['std'].values
df = pd.read_csv('../testdata/Pc_testset.csv')
# args.fold = df['molecular formula']为测试集
X_train = df.iloc[:, 2:].values
X_train = (X_train - mean) / std
model2d = joblib.load('../model/rf_pc_2d.pkl')
model3d = joblib.load('../model/rf_pc_3d.pkl')
y_pred_3d = model3d.predict(X_train)
y_pred_2d = model2d.predict(X_train[:, :76-11])
df['PC3D_pred'] = np.exp(y_pred_3d)
df['PC2D_pred'] = np.exp(y_pred_2d)
df = df[['Name', 'PC2D_pred', 'PC3D_pred']]
df.to_csv('./test_Pred/RF-PC.csv', index=False)